---
title: "Modelado y Pronóstico con LightGBM-Tweedie: Ruta El Silencio"
---

In [2]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import time

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import eda_toolkit as toolkit
from utils import build_features as bf
from utils import trainer_toolkit as tt
from utils import test_toolkit as ts

from eval.scorer import DemandScorer
from eval.holdout import build_holdout_dataset, evaluate_on_holdout
from models.baselines import seasonal_naive, historical_median
from models.lgbm_tweedie_wrapper import LightGBMTweedieWrapper
from trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

RESULTS_PATH = PROJECT_ROOT / "results" / "lgbm" / "silencio"
MODELS_PATH = PROJECT_ROOT / "models" / "silencio"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

train = pd.read_parquet(DATA_PROCESSED / "train_silencio.parquet")
holdout = pd.read_parquet(DATA_PROCESSED / "holdout_silencio.parquet")

feature_builder = lambda df: bf.build_features(df, dropna=True)

assert train.index.max() < holdout.index.min(), "Solapamiento temporal train/holdout"

print("=" * 70)
print(f"Ruta:    Silencio")
print(f"Train:   {len(train):>6} filas  ({train.index[0]} -> {train.index[-1]})")
print(f"Holdout: {len(holdout):>6} filas  ({holdout.index[0]} -> {holdout.index[-1]})")
print("=" * 70)

Ruta:    Silencio
Train:    56992 filas  (2023-01-01 04:30:00 -> 2025-05-26 17:15:00)
Holdout:  14248 filas  (2025-05-26 17:30:00 -> 2025-12-31 20:30:00)


In [2]:
train

,franja,anio,mes,dia,dia_semana,tipo_estacionalidad,nombre_evento,pasajeros,n_viajes
franja_ts,,,,,,,,,
2023-01-01 04:30:00,04:30:00,2023,1,1,7,FESTIVO_NACIONAL,Año Nuevo,0,0
2023-01-01 04:45:00,04:45:00,2023,1,1,7,FESTIVO_NACIONAL,Año Nuevo,0,0
2023-01-01 05:00:00,05:00:00,2023,1,1,7,FESTIVO_NACIONAL,Año Nuevo,0,1
2023-01-01 05:15:00,05:15:00,2023,1,1,7,FESTIVO_NACIONAL,Año Nuevo,70,1
2023-01-01 05:30:00,05:30:00,2023,1,1,7,FESTIVO_NACIONAL,Año Nuevo,103,2
...,...,...,...,...,...,...,...,...,...
2025-05-26 16:15:00,16:15:00,2025,5,26,1,DIAS_CORRIENTES,Dias corrientes del año,179,2
2025-05-26 16:30:00,16:30:00,2025,5,26,1,DIAS_CORRIENTES,Dias corrientes del año,210,2
2025-05-26 16:45:00,16:45:00,2025,5,26,1,DIAS_CORRIENTES,Dias corrientes del año,173,2


# Introducción

El Gradient Boosting con LightGBM es un marco de aprendizaje supervisado que construye un ensamble aditivo de árboles de decisión mediante la optimización secuencial de una función de pérdida diferenciable. A diferencia de los modelos que imponen una ecuación paramétrica para la evolución de la media condicional, LightGBM aprende la relación entre las variables predictoras y la respuesta directamente a partir de los datos, sin asumir una forma funcional específica.

En el contexto de la demanda de pasajeros por franja horaria, esta arquitectura ofrece una alternativa a los modelos recurrentes y a los enfoques econométricos tradicionales, al permitir que la relación entre el calendario, la estacionalidad, los rezagos y la demanda futura sea aprendida directamente de los datos. La naturaleza de la respuesta, sin embargo, impone una restricción sobre la función de pérdida: la serie presenta sobredispersión marcada en las horas de mayor operación y una proporción apreciable de franjas con cero pasajeros, estructura que una pérdida cuadrática o una verosimilitud Poisson subestimarían. Por esta razón se adopta la distribución Tweedie como familia de verosimilitud del objetivo, cuya forma compuesta Poisson-Gamma asigna masa puntual en cero y una densidad continua para los valores positivos.

El objetivo de esta sección es presentar, entrenar y evaluar modelos LightGBM con pérdida Tweedie para la predicción de la demanda de pasajeros por franja horaria en la ruta Silencio. Dado que la operación requiere pronósticos a múltiples escalas temporales, el problema se formula como una tarea de predicción directa multi-horizonte. Para cada intervalo operativo solicitado (30, 60 y 120 minutos) se entrena y optimiza un modelo LightGBM independiente, cada uno con su propio objetivo agregado y su propia validación. La evaluación sobre el *holdout* replica las condiciones de despliegue operativo mediante un reentrenamiento rodante por bloques con ventana deslizante. Este esquema respeta la partición cronológica y las restricciones metodológicas del proyecto: emplea exclusivamente variables disponibles en el instante de decisión y previene cualquier fuga de información entre conjuntos.

### Arquitectura

LightGBM construye un modelo aditivo de árboles de decisión que aproxima la función objetivo de forma secuencial. Para un conjunto de $N$ observaciones $\{\mathbf{x}_i, y_i\}_{i=1}^{N}$, el modelo final se expresa como la suma de $M$ árboles $h_m(\cdot)$ escalados por una tasa de aprendizaje $\nu$:

$$
F_M(\mathbf{x}_i) = \sum_{m=1}^{M} \nu \, h_m(\mathbf{x}_i),
$$

donde cada árbol $h_m \in \mathcal{H}$ es una función escalonada sobre el espacio de predictores. El ajuste es *forward stagewise*: en la iteración $m$, el modelo previo $F_{m-1}$ se mantiene fijo y se añade un nuevo árbol que corrige sus errores,

$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \, h_m(\mathbf{x}).
$$

El nuevo árbol se obtiene minimizando una expansión de segundo orden de la pérdida $\mathcal{L}$ alrededor de la predicción actual. Definiendo el gradiente $g_{im}$ y el hessiano $h_{im}$ de la pérdida respecto a $F_{m-1}(\mathbf{x}_i)$,

$$
g_{im} = \left. \frac{\partial \mathcal{L}(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}, \qquad
h_{im} = \left. \frac{\partial^2 \mathcal{L}(y_i, F)}{\partial F^2} \right|_{F = F_{m-1}(\mathbf{x}_i)},
$$

las hojas del árbol se calculan con la solución de Newton,

$$
w_{j}^{*} = - \frac{\sum_{i \in I_j} g_{im}}{\sum_{i \in I_j} h_{im} + \lambda},
$$

donde $I_j$ es el conjunto de observaciones que caen en la hoja $j$ y $\lambda$ es el término de regularización $\ell_2$. La ganancia de una partición que divide $I$ en $I_L$ e $I_R$ se evalúa como:

$$
\mathcal{G} = \frac{1}{2}\left[
\frac{\left(\sum_{i \in I_L} g_{im}\right)^2}{\sum_{i \in I_L} h_{im} + \lambda} +
\frac{\left(\sum_{i \in I_R} g_{im}\right)^2}{\sum_{i \in I_R} h_{im} + \lambda} -
\frac{\left(\sum_{i \in I} g_{im}\right)^2}{\sum_{i \in I} h_{im} + \lambda}
\right].
$$

Bajo el objetivo `tweedie`, LightGBM asume que la respuesta sigue una distribución Tweedie con parámetro de potencia $p$ y utiliza una función de enlace logarítmica, de modo que la media condicional se obtiene como $\hat{\mu}(\mathbf{x}) = \exp(F(\mathbf{x}))$. La pérdida que se minimiza es la desviación negativa de la log-verosimilitud Tweedie,

$$
\mathcal{L}_{\text{Tweedie}}(y, \hat{\mu}) = - \left[ y \, \frac{\hat{\mu}^{1-p}}{1-p} - \frac{\hat{\mu}^{2-p}}{2-p} \right].
$$

:::{admonition} Información: Distribución Tweedie
:class: info

La distribución Tweedie con parámetro de potencia $p$ pertenece a la familia exponencial y se caracteriza por la relación entre la media $\mu$ y la varianza:

$$
\operatorname{Var}(y) = \phi \, \mu^{p}, \qquad 1 < p < 2.
$$

Valores de $p$ cercanos a 1 aproximan la verosimilitud a una Poisson; valores cercanos a 2 la aproximan a una Gamma; los valores intermedios corresponden a la familia compuesta Poisson-Gamma, que admite masa en cero y una componente continua positiva. En este proyecto, el parámetro $p$ se estima empíricamente a partir de la relación varianza-media de la serie. El valor resultante se fija como constante durante el entrenamiento para garantizar la estabilidad y comparabilidad de la métrica.
:::

Con la función de enlace logarítmica, las derivadas de la pérdida respecto a la predicción del modelo son:

$$
\frac{\partial \mathcal{L}}{\partial F} = -y \, e^{(1-p)F} + e^{(2-p)F}, \qquad
\frac{\partial^2 \mathcal{L}}{\partial F^2} = -y(1-p) \, e^{(1-p)F} + (2-p) \, e^{(2-p)F}.
$$

Estas expresiones alimentan el gradiente $g_{im}$ y el hessiano $h_{im}$ en cada iteración del boosting y son las que permiten que el modelo pondere de forma diferenciada las franjas con cero pasajeros y las franjas de alta demanda. Para $1 < p < 2$, el hessiano es estrictamente positivo, lo que garantiza la convexidad local del subproblema de optimización en cada hoja.

El modelo completo $F_M(\cdot)$ asigna a cada vector de predictores $\mathbf{x}$ (hora, día de la semana, mes, codificación cíclica, estacionalidad, rezagos y tendencia temporal) una predicción de la media condicional de pasajeros acumulados en la ventana objetivo, $\hat{\mu}(\mathbf{x}) = \exp(F_M(\mathbf{x}))$.

### Protocolo de entrenamiento y validación

Se entrena un modelo LightGBM independiente por cada horizonte de predicción (30, 60 y 120 minutos, equivalentes a $h \in \{2, 4, 8\}$ franjas). El *target* de cada modelo es la suma rodante de pasajeros desde el instante de decisión hasta el final del intervalo, compartiendo todos el mismo conjunto de 23 variables predictoras. Dado que la agregación temporal altera la dispersión de la demanda, el parámetro de potencia $p$ de Tweedie se estima empíricamente de forma independiente para cada horizonte y se fija como constante durante su entrenamiento. Cada intervalo se predice en un único paso, sin usar las predicciones de un horizonte como insumo de otro.

Para mitigar el cambio de régimen estructural entre 2023 y 2025 y evitar sesgos por sobrepredicción, se implementó una ventana deslizante (*sliding window*). El hiperparámetro `window` define la historia reciente que observa el modelo; se exploraron tamaños entre 18,000 y 26,000 franjas (de 8 a 12 meses), con un calentamiento inicial fijado en 26,000.

La validación en `train_silencio.parquet` usa un esquema *walk-forward* de 5 pliegues, aplicando un *embargo* de $h-1$ franjas en la frontera para evitar fugas de información. Para la evaluación final, el `holdout_silencio.parquet` replica esta mecánica avanzando cronológicamente y reentrenando el modelo por bloques con las últimas `window` franjas disponibles.

La implementación utiliza `LightGBM` (v4.6) con el objetivo `tweedie`. El *early stopping* interno se evalúa sobre una partición cronológica 70/30 dentro de cada bloque de entrenamiento, respetando también el *embargo* del horizonte.

## Ventana de predicción de 30 minutos

### Entrenamiento

Para determinar el parámetro de potencia $p$ en la muestra de entrenamiento, se aplicó un estimador por momentos basado en la relación teórica entre la varianza y la media de la distribución Tweedie:

$$
\log(\operatorname{Var}(y)) = \log(\phi) + p \log(\mu)
$$

Al agrupar las observaciones por hora y día de la semana y calcular la pendiente de esta regresión lineal sobre las medias y varianzas resultantes, se obtuvo un valor de $p = 1.4$. Al contrastar este ajuste con los datos reales, el análisis visual reveló que la curva teórica no alcanzó la altura máxima del pico empírico y presentó desviaciones en los extremos del gráfico cuantil-cuantil. Una hipótesis plausible para explicar este comportamiento radicó en la alta heterogeneidad de la serie: al intentar abarcar la gran dispersión generada por la mezcla de regímenes operativos (horas pico, valles y fines de semana), la distribución teórica se aplanó en el centro y engrosó sus colas. No obstante, dicho sesgo marginal no comprometió el desempeño predictivo del ensamble LightGBM, ya que la función de pérdida Tweedie se aplicó de forma condicional sobre las hojas del árbol, capturando la dinámica local de cada franja en lugar del promedio global.

In [40]:
warmup_data = train["pasajeros"].iloc[:26000]
pasajeros_h2_warmup = warmup_data.rolling(window=2).sum().shift(-1).dropna()

p_warmup = tt._estimate_tweedie_p_internal(pasajeros_h2_warmup)

_, _ = tt.plot_tweedie_fit(
    series=pasajeros_h2_warmup, 
    p=p_warmup,
    titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
    xlab="Pasajeros acumulados en 30 min (h=2)",
    pads=(15, 15, 12), 
    legend_bbox=(1.0, 0.95)
)

Para la selección de hiperparámetros del modelo LightGBM correspondiente al horizonte de 30 minutos ($h = 2$ franjas), se implementó una validación cruzada temporal de cinco pliegues sobre el conjunto de entrenamiento. La búsqueda en Optuna exploró la complejidad de los árboles de decisión, los parámetros de regularización y el tamaño de la ventana de historia reciente.

El ensamble recibió directamente una matriz tabular con las 23 variables predictoras (calendario, codificación cíclica, estacionalidad, tendencia temporal y rezagos precalculados). Para evitar el sesgo de anticipación (*look-ahead bias*) y adaptarse al cambio de régimen estructural de la demanda, se implementó una estrategia de ventana deslizante (*sliding window*). En cada pliegue, el modelo se reentrenó con un bloque de historia delimitado por el hiperparámetro `window` y se validó con el bloque cronológico inmediatamente posterior, aplicando un *embargo* de una franja ($h - 1 = 1$) para garantizar la integridad temporal de los datos.

El espacio de búsqueda incluyó el tamaño de la ventana de entrenamiento (`window`), el número máximo de hojas por árbol (`num_leaves`), la tasa de aprendizaje (`learning_rate`), el mínimo de observaciones y de suma de hessianos por hoja (`min_data_in_leaf`, `min_sum_hessian_in_leaf`), las fracciones y frecuencia de submuestreo (`bagging_fraction`, `bagging_freq`, `feature_fraction`), así como las penalizaciones por regularización $\ell_1$ y $\ell_2$. El entrenamiento se configuró con un máximo de 5,000 árboles y una paciencia de 300 iteraciones para la parada temprana (*early stopping*) interna, evaluada sobre el último 10% cronológico de cada bloque de entrenamiento respetando también el embargo de una franja. El desempeño se midió mediante la desviación Tweedie, fijando el parámetro de potencia en $p = 1.4311$ (estimado exclusivamente con el bloque de calentamiento para evitar fuga de información) y aplicando un límite superior de 100 por observación para mitigar el impacto de eventos atípicos impredecibles.

In [6]:
WINDOW_MAX = 26000

def lgbm_search_space(trial: optuna.Trial) -> dict:
    return {
        "window": trial.suggest_categorical(
            "window", [18000, 20000, 22000, 24000, WINDOW_MAX]
        ),
        "num_leaves": trial.suggest_int("num_leaves", 15, 300, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0, step=0.05),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0, step=0.05),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 50.0, log=True),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf", 1e-3, 10.0, log=True
        ),
    }

exec_id = 8
exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=2,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=1,
    n_trials=100,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.1,
    model_kwargs={
        "tweedie_variance_power": 1.4311,
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": 8,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

Respecto a la eficiencia del algoritmo de búsqueda, la optimización se detuvo en el ensayo 58 sin alcanzar la totalidad de los 100 ensayos programados, debido a la activación del mecanismo de parada temprana (30 ensayos consecutivos sin mejora tras el ensayo 27). El mecanismo de poda (*pruning*) descartó 5 configuraciones, apoyando la utilidad del *Median Pruner* para abortar tempranamente arquitecturas con bajo desempeño predictivo durante las primeras particiones. De los 53 ensayos completados, la mejor configuración alcanzó una Desviación Tweedie de 3.2053.

El entrenamiento efectivo de los árboles a través de los 265 pliegues evaluados (53 ensayos completados $\times$ 5 pliegues) acumuló un tiempo neto de 30.43 minutos (1825.71 segundos), equivalente a un promedio de 6.38 segundos por pliegue bajo el esquema de ventana deslizante. La ejecución completa tuvo una duración de 35 minutos y 46 segundos (2,146 segundos); la diferencia de aproximadamente 5.34 minutos (320.29 segundos) respecto al tiempo acumulado de los pliegues obedeció al ajuste inicial sobre el bloque de calentamiento en cada ensayo y a la evaluación de las métricas de validación en memoria. Dado que LightGBM ejecutó la inferencia de forma vectorizada, la generación de pronósticos sobre el conjunto de prueba (*holdout*) se completó en fracciones de segundo en cada bloque rodante.

In [7]:
resumen_data = []

exec_id = 8
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name="LightGBMTweedieWrapper_h2_study", 
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
58,53,5,0,3.2053,1825.7100,6.3840


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una Desviación Tweedie en torno a 3.21. Dado que la variabilidad entre los pliegues de validación se mantuvo relativamente alta ($\sigma \approx 1.06$), las diferencias de milésimas entre estas configuraciones constituyeron un empate estadístico. No obstante, el proceso exhibió una notable estabilidad en la selección de hiperparámetros: los cinco mejores modelos convergieron de manera unánime hacia el límite superior de ventana de historia evaluado de 26,000 franjas, un submuestreo pronunciado de predictores (`feature_fraction` entre 0.40 y 0.45), fracciones moderadas de muestreo de filas (`bagging_fraction` entre 0.60 y 0.65), tasas de aprendizaje bajas y penalizaciones $\ell_1$ y $\ell_2$ cercanas a cero.

La selección consistente de 26000 franjas (equivalentes a más de un año calendario, dado que cada franja representa 15 minutos) permitió maximizar la cobertura del ciclo estacional anual y disponer de un mayor volumen de ejemplos históricos para estimar los patrones semanales y de calendario sin degradar el ajuste sobre los regímenes recientes. La combinación de tasas de aprendizaje bajas con un submuestreo estricto de variables y árboles de complejidad intermedia resultó suficiente para regularizar el ensamble y mitigar el sobreajuste sin requerir penalizaciones explícitas sobre los pesos de las hojas.

In [8]:
exec_dir = RESULTS_PATH / "exec_8"
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"26,000",42,17,0.4000,0.6500,0.0095,2.7e-04,2.2e-07,3.205 ± 1.058
"26,000",85,17,0.4000,0.6000,0.0086,5.3e-03,1.4e-06,3.216 ± 1.071
"26,000",34,14,0.4000,0.6500,0.0070,1.7e-01,4.7e-07,3.218 ± 1.064
"26,000",87,14,0.4000,0.6500,0.0067,6.7e-04,2.3e-06,3.221 ± 1.076
"26,000",30,11,0.4500,0.6500,0.0186,3.4e-03,8.3e-07,3.221 ± 1.048


La selección del modelo ganador entre las cinco mejores configuraciones se resolvió a favor de la primera (Desviación Tweedie de 3.205). Además de obtener el menor promedio de pérdida y una ligera reducción en la variabilidad entre pliegues frente a la segunda opción ($\sigma = 1.058$ frente a $\sigma = 1.071$), el modelo seleccionado ofreció un balance superior entre capacidad expresiva y parsimonia: redujo a menos de la mitad la complejidad estructural de los árboles (42 hojas frente a 85 hojas), mantuvo un submuestreo estricto de predictores (`feature_fraction` = 0.40) y empleó una tasa de aprendizaje ligeramente superior (0.0095 frente a 0.0086) que favoreció una convergencia eficiente sin sacrificar generalización.

In [22]:
best_params = {
    "window": 26000,
    "num_leaves": 42,
    "learning_rate": 0.009484004689773003,
    "min_data_in_leaf": 17,
    "feature_fraction": 0.4,
    "bagging_fraction": 0.65,
    "bagging_freq": 8,
    "lambda_l1": 0.00026621217695759153,
    "lambda_l2": 2.1604482856110947e-07,
    "min_sum_hessian_in_leaf": 0.0020565134495896358,
}

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": 1.4311,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": 8,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=RESULTS_PATH / "exec_8",
    feature_builder=feature_builder,
    horizon_slots=2,
    n_splits=5,
    warmup=26000,
    embargo=1,
    n_trials=100,
    model_kwargs={},
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (*In-Sample*) como en el de prueba (*Out-of-Sample*), a partir de las demandas predichas y las observaciones reales, se calcularon los residuos y su versión estandarizada mediante las siguientes expresiones:

$$
e_t = y_t - \hat{y}_t, \qquad z_t = \frac{e_t - \bar{e}}{s_e}
$$

donde $y_t$ es la demanda observada, $\hat{y}_t$ es la predicción del modelo, $\bar{e}$ es la media muestral de los residuos y $s_e$ es su desviación estándar.

Debido a que los estadísticos de Ljung-Box y ARCH-LM escalan linealmente con el tamaño de la muestra, evaluar la totalidad de la ventana activa de entrenamiento ($26000$ franjas) introduce un efecto de sobre-potencia (*over-power*) que infla los estadísticos ante cualquier dependencia residual. Para atenuar este sesgo sin alterar la secuencia cronológica de los rezagos, el diagnóstico se realizó sobre el último bloque continuo de entrenamiento ($N = 1000$ franjas) tomando observaciones no solapadas cada dos pasos ($h = 2$, resultando en $500$ bloques disjuntos de 30 minutos). Bajo esta especificación libre de superposición, las pruebas de Ljung-Box no detectaron autocorrelación significativa en ninguno de los rezagos evaluados tras aplicar la corrección de Holm $\left(p_{\text{ajustado}} \ge 0.067\right)$, mientras que las pruebas ARCH-LM rechazaron la hipótesis nula de homocedasticidad en todos los horizontes $\left(p_{\text{ajustado}} \le 0.014\right)$.

In [23]:
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
horizon_slots = 2
residuals_block = residuals_train[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),16.825,.078,.089,Sin Autocorrelación
Ljung-Box (Lag 20),34.621,.022,.067,Sin Autocorrelación
Ljung-Box (Lag 30),44.357,.044,.089,Sin Autocorrelación
ARCH-LM (Lag 10),26.661,.003,.009,Efectos ARCH Significativos
ARCH-LM (Lag 20),39.036,.007,.013,Efectos ARCH Significativos
ARCH-LM (Lag 30),49.460,.014,.014,Efectos ARCH Significativos


La ausencia de dependencia lineal significativa demuestra que el ensamble absorbió satisfactoriamente la dinámica media de corto y mediano plazo incluso sin incorporar rezagos intradía recientes ($k < 32$), confirmando que la persistencia aparente en evaluaciones previas obedecía al solapamiento mecánico entre ventanas contiguas de 30 minutos. Por su parte, la detección de efectos ARCH refleja la relación media-varianza propia de la demanda de pasajeros $\left(\operatorname{Var}(y) = \phi\,\mu^{1.4311}\right)$, donde la dispersión aumenta en las franjas de alta afluencia. Este comportamiento es característico de los conteos de transporte con sobredispersión, y la función de pérdida Tweedie logró asimilarlo sin comprometer la capacidad predictiva de la media condicional.

La inspección visual evidenció que el modelo LightGBM-Tweedie reprodujo los ciclos estacionales regulares de la operación de SOBUSA, las mesetas de demanda en días hábiles y los valles de fin de semana. También capturó las contracciones atípicas registradas entre 2024 y 2025, incluyendo las caídas de cierre de año, Semana Santa y la interrupción operativa de septiembre de 2024.

In [24]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2025-02-02", "2025-02-08"),
    zoom_right=("2025-02-28", "2025-02-28"),
    text_left="Febrero 2025",
    text_right="28 Febrero 2025",
    title="Demanda acumulada de pasajeros 2024-2025 (30 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12), top_ylim = (-2000, 40000),
    hspace=0.1, wspace = 0.05
)

A partir de febrero de 2025 se presentó una subestimación sistemática de la envolvente diaria: la demanda laboral real repuntó hacia niveles cercanos a $2.8 \times 10^4$ pasajeros diarios, mientras que la predicción se estabilizó alrededor de $2.3\text{--}2.4 \times 10^4$. Este desfase responde a la contracción estructural de la demanda documentada previamente en el análisis exploratorio (EDA) y a la amplitud de la ventana de entrenamiento ($26000$ franjas): al abarcar más de un año calendario, el ensamble pondera con mayor peso los regímenes más moderados de mediados y finales de 2024, lo que, sumado al suavizado hacia la media condicional propio de los árboles de decisión, ancla las predicciones de los primeros meses de 2025 por debajo del nuevo techo operativo.

En el detalle intradía de la semana del 2 al 8 de febrero de 2025 y de la jornada individual seleccionada, esta subestimación se concentró en la cúspide de los picos matutinos (alrededor de $800\text{--}930$ pasajeros por franja frente a máximos reales de $1050\text{--}1200$ entre el 3 y el 7 de febrero, y cerca de $570$ frente a más de $900$ en el detalle diario) y en las mesetas de media mañana y mediodía. A pesar de este suavizamiento en las crestas de alta frecuencia, la estructura basada en árboles capturó con precisión la transición matutina, el repunte vespertino y el descenso nocturno a lo largo de la serie.

### Evaluación

La evaluación del rendimiento del modelo LightGBM-Tweedie contrastó su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre demanda no vista. Mientras que la evaluación dentro de la muestra (*In-Sample*, sobre la ventana activa $w = 26000$ franjas del cierre de entrenamiento) verificó la convergencia del algoritmo mediante un ajuste estático, la evaluación fuera de la muestra (*Out-of-Sample*, conjunto *holdout* del `2025-05-26` al `2025-12-31`, $N = 14247$ franjas) determinó la utilidad operativa real ante nuevos regímenes de movilidad mediante reentrenamiento secuencial.

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios establecidos en el protocolo experimental. Estos criterios se basaron en el error relativo porcentual entre las métricas de prueba y entrenamiento, denotado como:

$$
\Delta_{\text{rel}} (\%) = \frac{M_{\text{test}} - M_{\text{train}}}{|M_{\text{train}}|} \times 100
$$

Se estableció un umbral de tolerancia relativo de $|\Delta_{\text{rel}}| = 5\%$ sobre la función de pérdida principal (*Tweedie Deviance*, con parámetro de potencia óptimo $p = 1.4311$). Un incremento relativo superior a este margen evidenciaría degradación en la capacidad de generalización.

En el contexto del pronóstico de demanda de pasajeros bajo verosimilitud Tweedie, debió considerarse la naturaleza asimétrica de esta función de pérdida y su sensibilidad diferenciada entre franjas de baja afluencia (incluyendo cierres operativos en cero) y horas pico. Dado que un desplazamiento puntual en la demanda real afectó de forma no lineal las métricas agregadas, la lectura de $\Delta_{\text{rel}}$ se complementó con métricas de escala (`MAE`, `RMSE`), métricas relativas (`SMAPE`, `MASE`), el diagnóstico de sesgo (`Bias`) y la inspección visual de las trayectorias diarias e intradía.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` de `LightGBMTweedieWrapper` sobre la serie continua concatenada (garantizando la disponibilidad exacta de los rezagos operativos sin pérdida de observaciones iniciales ni fuga de información). Este procedimiento aplicó un esquema de reentrenamiento periódico por bloques (`n_splits = 5`) con ventana deslizante fija (`window = 26,000`, `embargo = 1`), generando pronósticos puntuales en la escala original de pasajeros acumulados en ventanas de 30 minutos ($h = 2$). A diferencia del ajuste estático dentro de la muestra, la actualización progresiva de la ventana deslizante incorporó el repunte del primer semestre de 2025.

In [25]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_h2_30m_predictions.csv"
df_preds.to_csv(output_path, index=False)

El análisis de los residuos estandarizados no solapados fuera de muestra ($e_t = y_t - \hat{y}_t$, evaluado sobre los últimos $500$ bloques disjuntos de 30 minutos extraídos de las últimas $1000$ franjas del conjunto *holdout*) evidenció persistencia de estructura temporal y heterocedasticidad condicional en el error de pronóstico. Las pruebas de Ljung-Box rechazaron la hipótesis nula de independencia serial en todos los rezagos evaluados $\left(p_{\text{ajustado}} < 0.001\right)$, apoyando la presencia de autocorrelación remanente de corto y mediano plazo. De forma paralela, la prueba ARCH-LM detectó efectos ARCH altamente significativos en los tres horizontes evaluados $\left(p_{\text{ajustado}} < 0.001\right)$. Esto indica que, si bien el ensamble LightGBM-Tweedie captura con precisión la envolvente diaria e intradía de la demanda esperada, persisten fluctuaciones operativas de alta frecuencia y agrupamiento de varianza residual asociados a la dispersión propia de las franjas pico, los cierres atípicos de fin de año y la relación media-varianza del proceso $\left(\operatorname{Var}(y) = \phi\,\mu^{1.4311}\right)$.

In [26]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
horizon_slots = 2
residuals_block = residuals_test[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),159.456,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),325.344,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),375.488,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),51.825,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),63.220,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),76.300,< .001,< .001,Efectos ARCH Significativos


De acuerdo con los criterios establecidos en el protocolo experimental, no existió evidencia de sobreajuste (*overfitting*) ni de subajuste (*underfitting*) en el modelo LightGBM-Tweedie. La función de pérdida principal (`Tweedie Deviance`) permaneció prácticamente invariante entre entrenamiento y prueba (pasando de 2.2458 a 2.2471, un incremento relativo de apenas +0.06%), manteniéndose muy por debajo del umbral de tolerancia del 5% y respaldando la estabilidad del ensamble fuera de muestra. Este comportamiento se replicó en las métricas de error absoluto, cuadrático, mediano y relativo (`MAE`, `RMSE`, `MSE`, `Pinball` y `SMAPE`), las cuales registraron ligeras mejoras entre el 1.38% y el 2.77% en el conjunto de prueba.

Las mayores variaciones registradas en el `MASE` (+13.80%) y la inversión de signo del `Bias` (+224.66%) no obedecieron a un deterioro estructural del modelo, sino a los cambios de régimen entre ambos periodos y a la sensibilidad direccional frente a caídas puntuales de demanda. Mientras que en entrenamiento el sesgo negativo (-9.19 pasajeros) reflejó el anclaje histórico de la contracción de 2024 frente al repunte de inicios de 2025, en prueba la actualización de la ventana deslizante proyectó una envolvente laboral alta que sobrestimó transitoriamente el flujo esperado (+11.46 pasajeros) en vísperas de cierre de fin de año y días puente; simultáneamente, la menor dispersión del referente ingenuo estacional en el segundo semestre elevó el cociente `MASE` sin comprometer la precisión del régimen operativo regular.

In [27]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_h2_30m_predictions.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=1.4311,
    error_relativo=True,
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,2.2458,2.2471,+0.0606%
MAE,60.1543,59.0181,-1.8888%
RMSE,83.5272,82.3706,-1.3847%
MSE,6976.7880,6784.9095,-2.7502%
Bias,-9.1904,11.4566,+224.6585%
SMAPE,0.2320,0.2256,-2.7735%
MASE,0.6043,0.6877,+13.7992%
Pinball (τ=0.50),30.0771,29.5091,-1.8888%


La inspección visual de las trayectorias predichas frente a la demanda realizada en el conjunto *holdout* evidenció que el ensamble LightGBM-Tweedie reprodujo con alta fidelidad la envolvente diaria, la estacionalidad semanal y el perfil multimodal intradía a lo largo del segundo semestre. A escala agregada (panel superior), el ajuste siguió con precisión la demanda operativa en días hábiles y corrigió el sesgo a la baja registrado en la ventana de entrenamiento mediante el reentrenamiento deslizante, presentando únicamente episodios puntuales de sobreestimación asociados a caídas atípicas en días festivos (como finales de junio, julio y octubre) y cierres operativos de final de diciembre.

A escala de ventanas acumuladas de 30 minutos (paneles inferiores, correspondientes a la primera semana de septiembre y al 15 de octubre de 2025), persistió una ligera subestimación en la cúspide de la hora pico matutina (alrededor de las 05:00–05:30, alcanzando cerca de $1000\text{--}1200$ pasajeros acumulados frente a máximos reales de $1200\text{--},300$ en la ventana semanal, y cerca de $970$ frente a aproximadamente $1220$ el 15 de octubre), aunque de magnitud sustancialmente menor que la observada durante el entrenamiento. Este comportamiento respondió al efecto de suavizado propio de la estimación de la media condicional mediante árboles de decisión frente a fluctuaciones de alta frecuencia, sin comprometer la captura de la meseta de media mañana ni el repunte vespertino.

In [28]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda acumulada de pasajeros Holdout 2025 (30 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (30 min)",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)

## Ventana de predicción de 60 minutos

### Entrenamiento

Al aplicar el estimador por momentos sobre los conteos acumulados de 60 minutos ($h = 4$) en la ventana inicial de entrenamiento, se obtuvo un parámetro de potencia de $p = 1.8999$. Este desplazamiento hacia el límite superior de la familia Tweedie obedeció a la agregación de cuatro franjas consecutivas, que redujo los ceros puntuales y aproximó la distribución hacia un régimen continuo asimétrico. En la distribución teórica, la densidad suavizó la bimodalidad marginal entre fines de semana y días hábiles, mientras que el gráfico cuantil-cuantil mostró alineación en el cuerpo central con ligeras desviaciones en la cola superior asociadas a la capacidad física máxima de la flota.

In [41]:
warmup_data = train["pasajeros"].iloc[:26000]
pasajeros_h4_warmup = warmup_data.rolling(window=4).sum().shift(-3).dropna()

p_warmup = tt._estimate_tweedie_p_internal(pasajeros_h4_warmup)

_, _ = tt.plot_tweedie_fit(
    series=pasajeros_h4_warmup, 
    p=p_warmup,
    titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
    xlab="Pasajeros acumulados en 60 min (h=4)",
    pads=(15, 15, 12), 
    legend_bbox=(1.0, 0.95)
)

Para la selección de hiperparámetros del horizonte de 60 minutos ($h = 4$ franjas), se replicó el protocolo de validación cruzada temporal de cinco pliegues con ventana deslizante (*sliding window*) y el mismo espacio de búsqueda en Optuna utilizado en el horizonte previo. A diferencia del caso anterior, el exponente de varianza Tweedie se fijó en $p = 1.8999$ y el margen de *embargo* entre entrenamiento y validación se amplió a tres franjas ($h - 1 = 3$), evitando el solapamiento entre las ventanas acumuladas de una hora sin introducir fuga de información.

In [43]:
exec_id = 9
exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=4,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=3,
    n_trials=100,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.1,
    model_kwargs={
        "tweedie_variance_power": round(float(p_warmup), 4),
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": 8,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

La búsqueda finalizó en el ensayo 53 de los 100 programados al activarse la parada temprana tras 30 iteraciones consecutivas sin superar el óptimo registrado en el ensayo 22. Durante la exploración no se produjeron podas ni fallos, completándose las 53 configuraciones evaluadas con una Desviación Tweedie mínima de 0.4350.

El ajuste de los árboles a través de los 265 pliegues ejecutados (53 ensayos $\times$ 5 particiones) demandó un tiempo neto de 18.55 minutos (1112.72 segundos), equivalente a un promedio de 4.20 segundos por pliegue. La ejecución global tomó 21 minutos y 58 segundos (1,318 segundos); el margen restante de 3.42 minutos (205.28 segundos) correspondió al entrenamiento inicial sobre el bloque de calentamiento en cada iteración y al cómputo de las métricas de validación en memoria.

In [44]:
resumen_data = []

exec_id = 9
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name="LightGBMTweedieWrapper_h4_study",
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
53,53,0,0,0.4350,1112.7200,4.1990


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una Desviación Tweedie en torno a 0.44. Dado que la variabilidad entre los pliegues de validación se mantuvo elevada ($\sigma \approx 0.32$), las diferencias de milésimas entre estas alternativas constituyeron un empate estadístico. Asimismo, la exploración exhibió una convergencia estructural hacia la ventana máxima evaluada de 26000 franjas, árboles de complejidad controlada (entre 32 y 42 hojas), un submuestreo estricto de predictores (`feature_fraction` fijado de forma unánime en 0.40), fracciones altas de muestreo de filas (`bagging_fraction` entre 0.70 y 0.85), tasas de aprendizaje bajas y penalizaciones $\ell_1$ y $\ell_2$ cercanas a cero.

La elección consistente de 26000 franjas respaldó la conveniencia de abarcar más de un año calendario para capturar la estacionalidad anual y disponer de suficiente historial sobre los patrones semanales sin degradar la adaptación a los regímenes recientes. A su vez, el acoplamiento de tasas de aprendizaje reducidas con el submuestreo pronunciado de variables y árboles compactos resultó suficiente para regularizar el ensamble de forma implícita, haciendo innecesarias penalizaciones explícitas elevadas sobre los pesos de las hojas.

In [45]:
exec_dir = RESULTS_PATH / "exec_9"
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()
df_agg["std"] = df_agg["std"].fillna(0.0)

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"26,000",32,43,0.4000,0.8500,0.0202,2.8e-04,3.6e-07,0.435 ± 0.322
"26,000",39,43,0.4000,0.7000,0.0173,1.2e-02,1.5e-07,0.435 ± 0.321
"26,000",42,30,0.4000,0.8000,0.0178,1.6e-01,1.7e-08,0.436 ± 0.322
"26,000",42,41,0.4000,0.7000,0.0253,7.4e-01,1.1e-08,0.436 ± 0.319
"26,000",32,29,0.4000,0.8500,0.0134,1.9e-02,6.3e-08,0.437 ± 0.321


La selección del mejor modelo entre las cinco mejores configuraciones se resolvió a favor de la primera (Desviación Tweedie de 0.435). Con una variabilidad entre pliegues prácticamente idéntica a la de la segunda alternativa ($\sigma = 0.322$ frente a $\sigma = 0.321$), el modelo seleccionado ofreció un balance superior entre capacidad predictiva y parsimonia: adoptó una estructura de árboles más compacta (32 hojas frente a 39 hojas), mantuvo el submuestreo estricto de predictores (`feature_fraction` = 0.40), empleó una fracción de muestreo de observaciones más alta (`bagging_fraction` = 0.85 frente a 0.70) que estabilizó las estimaciones en cada iteración y utilizó una tasa de aprendizaje ligeramente superior (0.0202 frente a 0.0173) que agilizó la convergencia sin comprometer la capacidad de generalización.

In [3]:
exec_dir = RESULTS_PATH / "exec_9"
storage_url = f"sqlite:///{(exec_dir / 'optuna_study.db').resolve()}"
study = optuna.load_study(
    study_name="LightGBMTweedieWrapper_h4_study",
    storage=storage_url,
)
best_params = dict(study.best_params)

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": 1.8999,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": 8,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=4,
    n_splits=5,
    warmup=26000,
    embargo=3,
    n_trials=100,
    model_kwargs={},
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

Al evaluar los residuales estandarizados no solapados del bloque final de entrenamiento, las pruebas de diagnóstico indicaron que el modelo capturó de manera efectiva la dinámica de corto plazo. Durante las diez primeras horas operativas no se detectó dependencia lineal ni agrupamiento de volatilidad significativo tras aplicar la corrección de Holm.

En contraste, los rezagos vigésimo y trigésimo exhibieron autocorrelación y heterocedasticidad condicional vinculadas al ciclo diario de operación, dado que cada jornada efectiva comprende entre dieciséis y dieciocho bloques horarios consecutivos. Esta persistencia residual de media frecuencia es habitual en modelos de árboles impulsados por gradiente, los cuales priorizan la estimación condicional puntual sobre la modelación explícita de la varianza.

In [48]:
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
horizon_slots = 4
residuals_block = residuals_train[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),12.674,.242,.242,Sin Autocorrelación
Ljung-Box (Lag 20),60.386,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),68.874,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),10.697,.382,.382,Sin Efectos ARCH
ARCH-LM (Lag 20),61.275,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),49.602,.014,.027,Efectos ARCH Significativos


En el entrenamiento, la demanda predicha reprodujo con fidelidad el ciclo semanal y los cambios estacionales de nivel a lo largo de 2024, capturando tanto las caídas de fines de semana como los repuntes de agosto a noviembre. Al igual que en el horizonte de 30 minutos, se observó una tendencia sistemática a subestimar la demanda durante los primeros meses de 2025, donde la serie real alcanzó picos superiores a los 55,000 pasajeros acumulados mientras las predicciones se estabilizaron alrededor de 47,000.

Esta subestimación se detalló en los paneles inferiores correspondientes a febrero de 2025. En la ventana semanal del 2 al 8 de febrero, el ensamble logró replicar la forma de las ondas diarias y la desaceleración del sábado, pero atenuó la amplitud de los picos matutinos. En el perfil intradía del 28 de febrero, las estimaciones siguieron la trayectoria de la hora pico temprana y la meseta del mediodía por debajo de la curva observada, convergiendo hacia los valores reales únicamente en las horas de cierre vespertino.

In [51]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2025-02-02", "2025-02-08"),
    zoom_right=("2025-02-28", "2025-02-28"),
    text_left="Febrero 2025",
    text_right="28 Febrero 2025",
    title="Demanda acumulada de pasajeros 2024-2025 (60 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (60 min)",
    zoom_ylim_pad=(0.10, 0.12),
    top_ylim= (-2000, 80000), 
    hspace=0.1,
    wspace=0.05,
)

### Evaluación

Para el horizonte de 60 minutos ($h = 4$), se aplicó el mismo protocolo de evaluación comparativa y criterios de diagnóstico establecidos en la ventana previa, ajustando el parámetro de potencia de la función de pérdida Tweedie a $p = 1.8999$. Las predicciones fuera de muestra sobre el conjunto *holdout* se generaron mediante el esquema de reentrenamiento secuencial por bloques (`n_splits = 5`, `window = 26000`), ampliando el margen de *embargo* a tres franjas (`embargo = 3`) para evitar el solapamiento entre las ventanas acumuladas de una hora. De este modo, los pronósticos puntuales se obtuvieron directamente en la escala de pasajeros acumulados por bloque de 60 minutos, incorporando de forma progresiva el cambio de nivel registrado durante el primer semestre de 2025.

In [52]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_h4_60m_predictions.csv"
df_preds.to_csv(output_path, index=False)

En el diagnóstico de los residuales estandarizados no solapados del conjunto *holdout*, tanto las pruebas de Ljung-Box como las pruebas ARCH-LM resultaron estadísticamente significativas en todos los rezagos evaluados tras aplicar la corrección de Holm. A diferencia del cierre de entrenamiento, donde la dependencia serial y el agrupamiento de volatilidad de corto plazo permanecían contenidos, el desempeño fuera de muestra exhibió persistencia en los errores desde las primeras diez horas operativas.

Esta persistencia residual surgió de los desplazamientos de nivel y perturbaciones operativas no observadas durante el segundo semestre de 2025, que generaron rachas consecutivas de subestimación o sobreestimación entre bloques de reentrenamiento. Debido a que el ensamble estimó únicamente la media condicional puntual sin modelar explícitamente la estructura autorregresiva ni la heterocedasticidad del error, la varianza residual conservó memoria temporal en escenarios de demanda no estacionaria.

In [53]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
horizon_slots = 4
residuals_block = residuals_test[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),201.019,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),260.069,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),316.243,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),51.298,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),73.233,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),73.900,< .001,< .001,Efectos ARCH Significativos


El contraste entre entrenamiento y prueba descartó la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*) estructural. Aunque la variación relativa en la función de pérdida principal superó en magnitud absoluta el umbral de tolerancia del 5%, el cambio ocurrió en sentido favorable mediante una reducción sustancial del error en el conjunto de prueba. Este comportamiento se replicó en las métricas de escala y porcentaje simétrico, respaldando la capacidad de generalización del ensamble sobre demanda no vista.

La caída de la desviación Tweedie derivó de la sensibilidad de esta métrica cerca de cero bajo un exponente próximo al límite Gamma ($p = 1.8999$), dado que la ventana de entrenamiento incorporó eventos atípicos y caídas abruptas de afluencia que no se repitieron con la misma severidad durante el segundo semestre de 2025. El aumento relativo del `MASE` reflejó una mayor estabilidad de la serie en el periodo de prueba, lo cual redujo el error del referente ingenuo estacional; aun bajo este escenario, el ensamble conservó una mejora del 30% frente a dicho referente.

In [54]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_h4_60m_predictions.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=1.8999,
    error_relativo=True,
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,0.3577,0.1915,-46.4722%
MAE,95.1721,89.1646,-6.3123%
RMSE,137.4543,128.8554,-6.2558%
MSE,18893.6755,16603.7083,-12.1203%
Bias,-21.3720,17.8961,+183.7365%
SMAPE,0.1700,0.1596,-6.1493%
MASE,0.6226,0.7001,+12.4417%
Pinball (τ=0.50),47.5861,44.5823,-6.3123%


En cuanto al sesgo, el porcentaje elevado constituyó un artefacto aritmético producido por una inversión de signo alrededor de cero, al pasar de una leve sobreestimación promedio en entrenamiento a una leve subestimación fuera de muestra. En términos operativos, esta desviación media resultó marginal frente al volumen acumulado de pasajeros por ventana horaria y provino de la actualización periódica de pesos ante los cambios recientes de nivel.

Las predicciones en el conjunto de prueba evidenciaron que el reentrenamiento secuencial corrigió la subestimación observada al inicio de 2025 y siguió con precisión las variaciones semanales y los cambios estacionales de nivel entre junio y diciembre, presentando ligeras sobreestimaciones únicamente en semanas atípicas de octubre y en el cierre de diciembre. En la primera semana de septiembre, el modelo reprodujo sin desfase la amplitud de los picos matutinos cercanos a los 2,000 pasajeros por hora y la contracción operativa del fin de semana. De forma paralela, el perfil intradía del 15 de octubre respaldó la capacidad del ensamble para capturar la hora pico temprana, la meseta del mediodía alrededor de los 1,000 pasajeros acumulados y el descenso vespertino hacia el cierre de la jornada.

In [55]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda acumulada de pasajeros Holdout 2025 (60 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (60 min)",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)

## Ventana de predicción de 120 minutos

### Entrenamiento

Al aplicar el estimador por momentos sobre los conteos acumulados de 120 minutos ($h = 8$) en la ventana inicial de entrenamiento, se obtuvo un parámetro de potencia de $p = 1.99$. Esta convergencia hacia el límite superior de la familia Tweedie obedeció a la agregación de ocho franjas consecutivas, la cual eliminó la masa puntual en cero y aproximó la variable hacia una distribución Gamma continua y asimétrica. En la curva teórica, la densidad suavizó la bimodalidad marginal entre fines de semana y días hábiles, mientras que el gráfico cuantil-cuantil mostró alineación en el cuerpo central con ligeras desviaciones en la cola superior asociadas a la capacidad física máxima de la flota.

In [7]:
warmup_data = train["pasajeros"].iloc[:26000]
pasajeros_h8_warmup = warmup_data.rolling(window=8).sum().shift(-7).dropna()

p_warmup = tt._estimate_tweedie_p_internal(pasajeros_h8_warmup)

_, _ = tt.plot_tweedie_fit(
    series=pasajeros_h8_warmup,
    p=p_warmup,
    titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
    xlab="Pasajeros acumulados en 120 min (h=8)",
    pads=(15, 15, 12),
    legend_bbox=(1.0, 0.95)
)

Para la selección de hiperparámetros del horizonte de 120 minutos ($h = 8$ franjas), se replicó el protocolo de validación cruzada temporal de cinco pliegues con ventana deslizante (*sliding window*) y el mismo espacio de búsqueda en Optuna utilizado en los horizontes previos. En esta configuración, el exponente de varianza Tweedie se fijó en $p = 1.99$ y el margen de *embargo* entre entrenamiento y validación se amplió a siete franjas ($h - 1 = 7$), evitando el solapamiento entre las ventanas acumuladas de dos horas sin introducir fuga de información.

In [6]:
WINDOW_MAX = 26000

exec_id = 10
exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=8,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=7,
    n_trials=100,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.1,
    model_kwargs={
        "tweedie_variance_power": round(float(p_warmup), 4),
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": 8,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

La búsqueda finalizó en el ensayo 52 de los 100 programados al activarse la parada temprana tras 30 iteraciones consecutivas sin superar el óptimo registrado en el ensayo 21. Durante la exploración no se produjeron podas ni fallos, completándose las 52 configuraciones evaluadas con una Desviación Tweedie mínima de 0.3900.

El ajuste de los árboles a través de los 260 pliegues ejecutados (52 ensayos $\times$ 5 particiones) demandó un tiempo neto de 38.33 minutos (2,299.93 segundos), equivalente a un promedio de 8.85 segundos por pliegue. La ejecución global tomó 45 minutos y 27 segundos (2,727 segundos); el margen restante de 7.12 minutos (427.07 segundos) correspondió al entrenamiento inicial sobre el bloque de calentamiento en cada iteración y al cómputo de las métricas de validación en memoria.

In [9]:
resumen_data = []

exec_id = 10
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name="LightGBMTweedieWrapper_h8_study",
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
52,52,0,0,0.3900,2299.9300,8.8460


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una Desviación Tweedie media idéntica de 0.390. Dado que la variabilidad entre los pliegues de validación se mantuvo elevada ($\sigma \approx 0.492$), las diferencias submilimétricas entre estas alternativas constituyeron un empate estadístico. Asimismo, la exploración exhibió una convergencia estructural hacia la ventana máxima evaluada de 26,000 franjas, un umbral mínimo elevado de observaciones por hoja (`min_data_in_leaf` entre 106 y 150), un submuestreo estricto de predictores (`feature_fraction` entre 0.40 y 0.45), fracciones moderadas de muestreo de filas (`bagging_fraction` entre 0.55 y 0.65), tasas de aprendizaje muy reducidas (entre 0.0013 y 0.0054) y penalizaciones $\ell_1$ y $\ell_2$ moderadas a bajas.

La elección consistente de 26,000 franjas respaldó la conveniencia de abarcar más de un año calendario para capturar la estacionalidad anual y disponer de suficiente historial sobre los patrones semanales sin degradar la adaptación a los regímenes recientes. A su vez, el acoplamiento de tasas de aprendizaje muy bajas con el submuestreo pronunciado de variables y el elevado soporte mínimo por hoja resultó fundamental para estabilizar el ensamble en un horizonte de dos horas, previniendo el sobreajuste incluso en configuraciones con mayor número de ramificaciones.

In [10]:
exec_dir = RESULTS_PATH / "exec_10"
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()
df_agg["std"] = df_agg["std"].fillna(0.0)

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"26,000",80,147,0.4000,0.6500,0.0037,7.7e-02,2.0e-02,0.390 ± 0.492
"26,000",166,106,0.4000,0.5500,0.0025,1.4e-03,2.6e-02,0.390 ± 0.492
"26,000",189,122,0.4000,0.6500,0.0054,5.9e-03,6.0e-02,0.390 ± 0.492
"26,000",215,150,0.4000,0.6000,0.0038,8.6e-03,6.3e-02,0.390 ± 0.491
"26,000",202,115,0.4500,0.6500,0.0013,6.9e-03,6.6e-04,0.390 ± 0.492


La selección del mejor modelo entre las cinco mejores configuraciones se resolvió a favor de la primera (Desviación Tweedie de 0.390). Con una variabilidad entre pliegues idéntica a la de la segunda alternativa ($\sigma = 0.492$), el modelo seleccionado ofreció un balance superior entre capacidad predictiva y parsimonia: adoptó una estructura de árboles sustancialmente más compacta (80 hojas frente a 166 hojas), mantuvo el submuestreo estricto de predictores (`feature_fraction` = 0.40), empleó una fracción de muestreo de observaciones más alta (`bagging_fraction` = 0.65 frente a 0.55) y un mayor soporte mínimo por hoja (147 frente a 106 observaciones) que estabilizaron las estimaciones en cada iteración, incorporó una regularización $\ell_1$ más estricta (`7.7e-02` frente a `1.4e-03`) y utilizó una tasa de aprendizaje ligeramente superior (0.0037 frente a 0.0025) que agilizó la convergencia sin comprometer la capacidad de generalización.

In [6]:
exec_dir = RESULTS_PATH / "exec_10"
storage_url = f"sqlite:///{(exec_dir / 'optuna_study.db').resolve()}"
study = optuna.load_study(
    study_name="LightGBMTweedieWrapper_h8_study",
    storage=storage_url,
)
best_params = dict(study.best_params)

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": 1.99,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": 8,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=8,
    n_splits=5,
    warmup=26000,
    embargo=7,
    n_trials=100,
    model_kwargs={},
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

En el diagnóstico de los residuales estandarizados no solapados del bloque final de entrenamiento, el incremento en la escala de agregación modificó el comportamiento de los errores respecto a los horizontes más cortos. Las pruebas ARCH-LM no aportaron evidencia de agrupamiento de volatilidad en ninguno de los rezagos evaluados tras aplicar la corrección de Holm $\left(p_{\text{ajustado}} \ge 0.222\right)$, un resultado consistente con la hipótesis de que la agregación acumulada de ocho franjas suaviza las fluctuaciones puntuales de alta frecuencia y estabiliza la varianza local del error.

Por el contrario, el estadístico de Ljung-Box resultó significativo en todos los rezagos evaluados $\left(p_{\text{ajustado}} \le 0.002\right)$, revelando persistencia lineal desde el décimo rezago (equivalente a veinte horas operativas). Al pronosticar bloques acumulados de dos horas bajo un esquema de árboles impulsados por gradiente sin términos autorregresivos explícitos sobre el error, las desviaciones sostenidas durante los cambios de régimen introducen dependencia serial entre ventanas sucesivas.

In [12]:
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
horizon_slots = 8
residuals_block = residuals_train[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),30.713,< .001,.001,Autocorrelación Significativa
Ljung-Box (Lag 20),43.979,.002,.002,Autocorrelación Significativa
Ljung-Box (Lag 30),63.132,< .001,.001,Autocorrelación Significativa
ARCH-LM (Lag 10),17.015,.074,.222,Sin Efectos ARCH
ARCH-LM (Lag 20),20.204,.445,.585,Sin Efectos ARCH
ARCH-LM (Lag 30),33.716,.292,.585,Sin Efectos ARCH


En la muestra de entrenamiento, el ensamble replicó adecuadamente la estacionalidad semanal y las transiciones de nivel registradas a lo largo de 2024, ajustándose tanto a las caídas de los fines de semana como a los incrementos de afluencia entre agosto y noviembre. No obstante, al inicio de 2025 la subestimación observada en ventanas previas se acentuó de manera marcada: mientras la demanda acumulada real alcanzó máximos superiores a los 110,000 pasajeros, las predicciones quedaron contenidas alrededor de los 92,000 entre enero y marzo, reduciendo esa brecha recién hacia abril y mayo.

Los paneles inferiores detallaron esta brecha en febrero de 2025. Durante la semana del 2 al 8 de febrero, el modelo conservó la morfología de los ciclos diarios y la contracción del sábado, pero recortó sustancialmente el techo de los picos matutinos, situando sus máximos en torno a los 2500–2700 pasajeros frente a los más de 3,000 observados. En el detalle intradía del 28 de febrero, la curva estimada se mantuvo sistemáticamente por debajo de la serie real desde la apertura hasta el final de la tarde, con una brecha cercana a los 1000 pasajeros en el pico temprano, alcanzando los valores observados únicamente después de las 19:00 horas.

In [14]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2025-02-02", "2025-02-08"),
    zoom_right=("2025-02-28", "2025-02-28"),
    text_left="Febrero 2025",
    text_right="28 Febrero 2025",
    title="Demanda acumulada de pasajeros 2024-2025 (120 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (120 min)",
    zoom_ylim_pad=(0.10, 0.12),
    top_ylim= (-2000, 180000), 
    hspace=0.1,
    wspace=0.05,
)

### Evaluación

Para el horizonte de 120 minutos ($h = 8$), se aplicó el mismo protocolo de evaluación comparativa y criterios de diagnóstico establecidos en las ventanas previas, fijando el parámetro de potencia de la función de pérdida Tweedie en $p = 1.99$. Las predicciones fuera de muestra sobre el conjunto *holdout* se generaron mediante el esquema de reentrenamiento secuencial por bloques (`n_splits = 5`, `window = 26000`), ampliando el margen de *embargo* a siete franjas (`embargo = 7`) para evitar el solapamiento entre las ventanas acumuladas de dos horas. De este modo, los pronósticos puntuales se obtuvieron directamente en pasajeros acumulados por bloque de 120 minutos, incorporando de forma progresiva el cambio de nivel registrado durante el primer semestre de 2025.

In [15]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_h8_120m_predictions.csv"
df_preds.to_csv(output_path, index=False)

En el diagnóstico de los residuales estandarizados no solapados del conjunto *holdout*, tanto las pruebas de Ljung-Box $\left(p_{\text{ajustado}} < 0.001\right)$ como las pruebas ARCH-LM $\left(p_{\text{ajustado}} \le 0.016\right)$ resultaron estadísticamente significativas en todos los rezagos evaluados tras aplicar la corrección de Holm, replicando el comportamiento observado en los horizontes previos. A diferencia del cierre de entrenamiento, donde la varianza local permaneció contenida, la evaluación fuera de muestra exhibió tanto autocorrelación como agrupamiento de volatilidad desde el décimo rezago (equivalente a veinte horas operativas).

Esta dinámica residual surgió de los desplazamientos de nivel y perturbaciones operativas no observadas durante el segundo semestre de 2025, que generaron rachas consecutivas de subestimación o sobreestimación entre bloques de reentrenamiento. Debido a que el ensamble estimó únicamente la media condicional puntual sin modelar explícitamente la estructura autorregresiva ni la heterocedasticidad del error, la varianza residual conservó memoria temporal en escenarios de demanda no estacionaria.

In [16]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
horizon_slots = 8
residuals_block = residuals_test[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),81.650,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),116.945,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),122.017,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),49.192,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),48.914,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),49.010,.016,.016,Efectos ARCH Significativos


El contraste entre entrenamiento y prueba descartó la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*) estructural. Aunque la variación relativa en la función de pérdida principal superó en magnitud absoluta el umbral de tolerancia del 5%, el cambio ocurrió en sentido favorable mediante una reducción del 77.75% en el conjunto de prueba (pasando de 0.4190 a 0.0932). Este comportamiento se replicó de manera consistente en las métricas de escala y porcentaje simétrico, con reducciones del 3.28% en el MAE, del 5.38% en el RMSE y del 2.08% en el sMAPE, respaldando la capacidad de generalización del ensamble sobre demanda no vista.

La caída pronunciada de la desviación Tweedie resultó consistente con la hipótesis de suavizamiento generada por la agregación temporal de ocho franjas, combinada con la sensibilidad inversa de esta métrica cerca de cero bajo un exponente prácticamente Gamma $\left(p = 1.99\right)$. Al acumular bloques de dos horas, se eliminaron los ceros operativos y se atenuaron los valles abruptos de afluencia que penalizaban fuertemente la ventana de entrenamiento, produciendo una serie más regular en el segundo semestre de 2025. Por su parte, el aumento relativo del 18.60% en el MASE (de 0.6225 a 0.7383) reflejó esa misma estabilización de la serie en el periodo de prueba, la cual redujo el error del referente ingenuo estacional; aun bajo este escenario, el ensamble conservó una mejora superior al 26% frente a dicho referente.

In [17]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_h8_120m_predictions.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=1.99,
    error_relativo=True,
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,0.4190,0.0932,-77.7531%
MAE,153.1815,148.1644,-3.2753%
RMSE,232.8877,220.3626,-5.3782%
MSE,54236.6937,48559.6695,-10.4671%
Bias,-38.1536,44.4156,+216.4126%
SMAPE,0.1331,0.1303,-2.0814%
MASE,0.6225,0.7383,+18.6018%
Pinball (τ=0.50),76.5907,74.0822,-3.2753%


Las predicciones en el conjunto de prueba evidenciaron que el esquema de reentrenamiento secuencial mitigó gran parte de la subestimación registrada al inicio de 2025 y siguió las variaciones semanales y los cambios estacionales de nivel entre junio y diciembre. A escala macroscópica persistió una leve tendencia a subestimar los techos de demanda acumulada en varios meses del segundo semestre, presentando una brecha ligeramente más visible que en los horizontes más cortos.

Al descender a la resolución operativa de corto plazo, el ensamble reprodujo con alta fidelidad la estructura de la demanda. Durante la primera semana de septiembre, el modelo capturó sin desfase la amplitud de los picos matutinos y la contracción operativa del fin de semana. El perfil intradía del 15 de octubre respaldó la precisión del ajuste diario, replicando casi de forma exacta la subida de la hora pico temprana, la meseta del mediodía y el descenso vespertino hacia el cierre de la jornada.

In [18]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda acumulada de pasajeros Holdout 2025 (120 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (120 min)",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)

## Conclusiones

La implementación y evaluación del ensamble LightGBM con función de pérdida Tweedie permitieron contrastar su capacidad predictiva directa a través de tres horizontes operativos de agregación ($h \in \{2, 4, 8\}$, equivalentes a 30, 60 y 120 minutos) sobre la demanda de pasajeros de la ruta Silencio. A diferencia de los enfoques recursivos, la formulación independiente por horizonte demostró una sólida capacidad de generalización fuera de muestra, descartando el sobreajuste en todos los escenarios al mantener las variaciones relativas dentro de los márgenes aceptables y superar consistentemente al referente ingenuo estacional con mejoras superiores al 26% en el `MASE`.

El incremento en el horizonte de agregación modificó sustancialmente la distribución estadística del objetivo y el comportamiento de los errores. Al pasar de ventanas de 30 minutos a bloques acumulados de 60 y 120 minutos, la suma de franjas consecutivas eliminó los ceros operativos y suavizó la dispersión de alta frecuencia, desplazando el parámetro de potencia Tweedie desde un régimen compuesto Poisson-Gamma ($p = 1.4311$ en $h = 2$) hacia el límite continuo Gamma ($p = 1.8999$ en $h = 4$ y $p = 1.99$ en $h = 8$). Esta regularización natural inducida por la agregación temporal redujo drásticamente la desviación Tweedie fuera de muestra (cayendo de `2.2471` en 30 minutos a `0.0932` en 120 minutos) y disipó el agrupamiento de volatilidad de corto plazo en el cierre de entrenamiento, aunque aumentó la inercia de los errores ante cambios sostenidos de régimen.

El protocolo de optimización en Optuna exhibió una convergencia estructural consistente hacia el límite superior de historial disponible (`window = 26000` franjas, equivalente a más de un año calendario) y un submuestreo estricto de variables (`feature_fraction` = 0.40) en los tres horizontes. Esta selección confirmó que abarcar un ciclo anual completo resulta indispensable para capturar la estacionalidad semanal y de calendario sin degradar la adaptación reciente, mientras que el acoplamiento de tasas de aprendizaje reducidas con un alto soporte mínimo de observaciones por hoja y árboles parsimoniosos (entre 32 y 80 hojas) bastó para regularizar el ensamble en escalas temporales amplias.

Finalmente, la arquitectura de árboles impulsados por gradiente capturó con alta precisión la morfología intradía, la transición de los picos matutinos, la meseta del mediodía y la contracción operativa de los fines de semana. Al estimar exclusivamente la media condicional puntual mediante particiones escalonadas, los modelos exhibieron una tendencia a suavizar las crestas extremas de demanda durante transiciones estructurales abruptas como el repunte de inicios de 2025. Si bien el esquema de reentrenamiento secuencial por bloques mitigó este sesgo a la baja en el conjunto de prueba, la persistencia de autocorrelación residual fuera de muestra evidenció que la ausencia de términos autorregresivos explícitos sobre el error limita la velocidad de ajuste inmediato ante perturbaciones operativas no estacionarias.